In [1]:
%use kandy

We are trying to estimate the performance ratio of stdlib functions in Kotlin/Native vs. JVM. For this example, we focus
on `String.substring` method used in this particular scenario. We need to exclude the effect of GC to get the performace
ratio of execution time of the function itself. The following cell describes, how we collect the data. We
end up with a list of execution times.

In [2]:
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardOpenOption

val kotlincNative = System.getProperty("user.home") + "/.konan/kotlin-native-prebuilt-linux-x86_64-2.2.10/bin/kotlinc-native"
val tempDir = Files.createTempDirectory("knative-perf")

fun runProcess(vararg command: String): String =
    ProcessBuilder(command.toList())
        .directory(tempDir.toFile())
        .start()
        .let { process ->
            process.inputStream.bufferedReader().readText()
                .also {
                    if (process.waitFor() != 0) {
                        process.errorStream.bufferedReader().readText().let(::println)
                        throw Exception("Process failed with code ${process.exitValue()}")
                    }
                }
        }

val source = """
import kotlin.concurrent.Volatile
import kotlin.time.measureTime
import kotlin.time.DurationUnit
import kotlin.time.Duration

@Volatile
private var blackhole: Any? = null
@Volatile
private var string = "abc".repeat(100000)

fun main() {
    repeat(1_000_000) { blackhole = string.substring(1000, 5000) }
    val iterations = 10_000
    val times = ArrayList<Duration>(iterations)
    repeat(iterations) {
        measureTime {
            repeat(100) {
                blackhole = string.substring(1000, 5000)
            }
        }.let { times.add(it) }
    }
    times.forEach { println(it.toDouble(DurationUnit.SECONDS)) }
}
"""

val sourceFile = tempDir.resolve("benchmark.kt")
val outputBinary = tempDir.resolve("benchmark.kexe")

Files.writeString(
    sourceFile,
    source,
    StandardOpenOption.CREATE,
)

runProcess(
        kotlincNative,
        sourceFile.toString(),
        "-o", outputBinary.toString(),
        "-Xgc=noop"
)

val times = runProcess(outputBinary.toString())
    .split("\n")
    .mapNotNull { it.takeIf { it.isNotEmpty() }?.toDouble() }
    .toList()

For the exclusion of GC, we assume that the execution time distribution is bimodal (i .e. there are two peaks). The
first peak is formed by the executions that didn't trigger GC. The second peak is formed by the executions that
triggered GC. By finding a threshold for the execution time that lies between the two peaks, we can separate the
executions into those that ran GC and those which didn't.

In [3]:
val threshold = 0.00014

In [4]:
plot {
    histogram(times, binsOption = BinsOption.byNumber(100))
    vLine {
        xIntercept.constant(threshold)
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="MVRlY4"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("MVRlY4");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"x",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count"
},
"stat":"identity",
"data":{
"x":[1.2232043999999998E-4,1.2351965999999999E-4,1.2471888E-4,1.259181E-4,1.2711732E-4,1.2831654E-4,1.2951576E-4,1.3071498E-4,1.319142E-4,1.3311342E-4,1.3431264E-4,1.3551186E-4,1.3671108E-4,1.379103E-4,1.3910951999999998E-4,1.4030873999999998E-4,1.4150795999999998E-4,1.4270717999999999E-4,1.439064E-4,1.4510562E-4,1.4630484E-4,1.4750406E-4,1.4870328E-4,1.499025E-4,1.5110172E-4,1.5230094E-4,1.5350016E-4,1.5469938E-4,1.558986E-4,1.5709782E-4,1.5829704E-4,1.5949626E-4,1.6069548E-4,1.6189470000000001E-4,1.6309392000000002E-4,1.6429314E-4,1.6549236E-4,1.6669158E-4,1.678908E-4,1.6909002E-4,1.7028924E-4,1.7148846E-4,1.7268768E-4,1.738869E-4,1.7508612E-4,1.7628533999999998E-4,1.7748455999999998E-4,1.7868377999999998E-4,1.7988299999999999E-4,1.8108222E-4,1.8228144E-4,1.8348066E-4,1.8467988E-4,1.858791E-4,1.8707832E-4,1.8827754E-4,1.8947676E-4,1.9067598E-4,1.918752E-4,1.9307442E-4,1.9427364E-4,1.9547286E-4,1.9667208E-4,1.978713E-4,1.9907052E-4,2.0026974000000001E-4,2.0146896000000002E-4,2.0266818000000002E-4,2.0386740000000002E-4,2.0506662E-4,2.0626584E-4,2.0746506E-4,2.0866428E-4,2.098635E-4,2.1106272E-4,2.1226193999999998E-4,2.1346115999999998E-4,2.1466037999999998E-4,2.1585959999999998E-4,2.1705881999999998E-4,2.1825803999999999E-4,2.1945726E-4,2.2065648E-4,2.218557E-4,2.2305492E-4,2.2425414E-4,2.2545336E-4,2.2665258E-4,2.278518E-4,2.2905102E-4,2.3025024E-4,2.3144946E-4,2.3264868E-4,2.338479E-4,2.3504712E-4,2.3624634E-4,2.3744556E-4,2.3864478000000001E-4,2.3984400000000002E-4,2.4104322000000002E-4],
"count":[43.0,382.0,1035.0,1210.0,1234.0,1047.0,789.0,628.0,432.0,312.0,203.0,133.0,102.0,70.0,47.0,22.0,31.0,53.0,69.0,86.0,72.0,97.0,67.0,76.0,63.0,62.0,63.0,44.0,59.0,35.0,31.0,32.0,59.0,93.0,139.0,142.0,142.0,126.0,108.0,78.0,65.0,63.0,33.0,30.0,30.0,19.0,12.0,6.0,8.0,14.0,24.0,22.0,11.0,17.0,15.0,15.0,11.0,7.0,7.0,11.0,7.0,5.0,3.0,8.0,5.0,6.0,1.0,3.0,3.0,3.0,4.0,2.0,3.0,1.0,0.0,2.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
}]
}
},{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"xintercept":1.4E-4,
"geom":"vline",
"data":{
}
}],
"spec_id":"2"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observ

We then estimate the total time spent in GC by the following formula.

In [5]:
val (noGC, GC) = times.partition { it < threshold }
val avgNoGC = noGC.average()
GC.sumOf { it - avgNoGC } / times.sum()

0.057730701715271164

For measurements that run at least 100 000 iterations and don't print the times throught the execution (but all at once at the end), we estimate that GC is responsible for over 80% of the execution. That is insane. Therefore, we question the results.

* Can GC in K/N be responsible for such a high percentage of execution time? After all, this is very memory heavy benchmark.
* Is there an error in the benchmark setup, collection of the data or the processing?
* Is any one of our assumptions incorrect?
* Is this method for estimating GC time sound?